In [1]:
!nvidia-smi

Tue Jun  9 14:49:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q PyMuPDF langdetect bitsandbytes transformers peft accelerate datasets tqdm

In [4]:
import fitz
import langdetect
import bitsandbytes
import transformers
import peft
print("✅ All libraries ready!")

✅ All libraries ready!


In [5]:
import os

path = '/kaggle/input/datasets/deepthynarayanan/fin-docs'
for f in os.listdir(path):
    print(f)

10-Q4-2024-As-Filed.pdf
Global Economic Prospects January 2026.pdf
GEP-Jan-2024.pdf
2024_Annual_Report.docx
tsla-20241231-gen.pdf
01LETTER290526A5A73B174EA340DA8C383A5D58FF0825.pdf
0AR29052026F5B979AF274E445ABB1593EB226906335.pdf
12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.pdf


In [7]:
import fitz  # PyMuPDF
import os

# Define paths
PDF_PATH = '/kaggle/input/datasets/deepthynarayanan/fin-docs'
OUTPUT_PATH = '/kaggle/working/domain_corpus'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Get all PDFs
pdf_files = [f for f in os.listdir(PDF_PATH) if f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDFs:")
for f in pdf_files:
    print(f"  - {f}")

Found 7 PDFs:
  - 10-Q4-2024-As-Filed.pdf
  - Global Economic Prospects January 2026.pdf
  - GEP-Jan-2024.pdf
  - tsla-20241231-gen.pdf
  - 01LETTER290526A5A73B174EA340DA8C383A5D58FF0825.pdf
  - 0AR29052026F5B979AF274E445ABB1593EB226906335.pdf
  - 12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.pdf


In [8]:
import fitz
import os

# Extract text from each PDF and save as .txt
print("Extracting text from PDFs...\n")

extraction_stats = []

for pdf_file in pdf_files:
    pdf_path = os.path.join(PDF_PATH, pdf_file)
    txt_filename = pdf_file.replace('.pdf', '.txt')
    txt_path = os.path.join(OUTPUT_PATH, txt_filename)
    
    try:
        doc = fitz.open(pdf_path)
        num_pages = len(doc)
        
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        
        # Save to .txt file
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(full_text)
        
        word_count = len(full_text.split())
        print(f"✅ {pdf_file}")
        print(f"   Pages: {num_pages} | Words: {word_count:,}")
        
        extraction_stats.append({
            'file': txt_filename,
            'pages': num_pages,
            'words': word_count
        })
        
    except Exception as e:
        print(f"❌ {pdf_file} — Error: {e}")

print(f"\n📊 Total files extracted: {len(extraction_stats)}")

Extracting text from PDFs...

✅ 10-Q4-2024-As-Filed.pdf
   Pages: 121 | Words: 64,170
✅ Global Economic Prospects January 2026.pdf
   Pages: 244 | Words: 136,498
✅ GEP-Jan-2024.pdf
   Pages: 230 | Words: 125,215
✅ tsla-20241231-gen.pdf
   Pages: 144 | Words: 71,876
✅ 01LETTER290526A5A73B174EA340DA8C383A5D58FF0825.pdf
   Pages: 1 | Words: 0
✅ 0AR29052026F5B979AF274E445ABB1593EB226906335.pdf
   Pages: 246 | Words: 100,256
✅ 12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.pdf
   Pages: 28 | Words: 12,208

📊 Total files extracted: 7


In [9]:
!pip install langdetect -q

In [10]:
from langdetect import detect
import os

print("=" * 60)
print("CLEANING PIPELINE")
print("=" * 60)

# --- BEFORE stats ---
txt_files = os.listdir(OUTPUT_PATH)
print(f"\n📊 BEFORE cleaning: {len(txt_files)} files")

# --- Step 1: Length Filter (min 1000 words) ---
print("\n🔍 Step 1: Length Filter (minimum 1000 words)")
removed_length = []
for f in txt_files:
    path = os.path.join(OUTPUT_PATH, f)
    with open(path, 'r', encoding='utf-8') as file:
        words = len(file.read().split())
    if words < 1000:
        os.remove(path)
        removed_length.append(f)
        print(f"  ❌ Removed: {f} ({words} words)")

print(f"  Removed {len(removed_length)} files")

# --- Step 2: Deduplication (by file hash) ---
print("\n🔍 Step 2: Deduplication (file hashing)")
import hashlib
seen_hashes = set()
removed_dupes = []
for f in os.listdir(OUTPUT_PATH):
    path = os.path.join(OUTPUT_PATH, f)
    with open(path, 'rb') as file:
        file_hash = hashlib.md5(file.read()).hexdigest()
    if file_hash in seen_hashes:
        os.remove(path)
        removed_dupes.append(f)
        print(f"  ❌ Removed duplicate: {f}")
    else:
        seen_hashes.add(file_hash)

print(f"  Removed {len(removed_dupes)} files")

# --- Step 3: Language Filter (English only) ---
print("\n🔍 Step 3: Language Filter (English only)")
removed_lang = []
for f in os.listdir(OUTPUT_PATH):
    path = os.path.join(OUTPUT_PATH, f)
    with open(path, 'r', encoding='utf-8') as file:
        sample = file.read(500)  # check first 500 chars
    try:
        lang = detect(sample)
        if lang != 'en':
            os.remove(path)
            removed_lang.append(f)
            print(f"  ❌ Removed non-English: {f} (detected: {lang})")
    except:
        pass

print(f"  Removed {len(removed_lang)} files")

# --- AFTER stats ---
remaining = os.listdir(OUTPUT_PATH)
print(f"\n📊 AFTER cleaning: {len(remaining)} files")
print("\n✅ Final corpus:")
for f in remaining:
    path = os.path.join(OUTPUT_PATH, f)
    with open(path, 'r', encoding='utf-8') as file:
        words = len(file.read().split())
    print(f"  - {f} ({words:,} words)")

CLEANING PIPELINE

📊 BEFORE cleaning: 7 files

🔍 Step 1: Length Filter (minimum 1000 words)
  ❌ Removed: 01LETTER290526A5A73B174EA340DA8C383A5D58FF0825.txt (0 words)
  Removed 1 files

🔍 Step 2: Deduplication (file hashing)
  Removed 0 files

🔍 Step 3: Language Filter (English only)
  Removed 0 files

📊 AFTER cleaning: 6 files

✅ Final corpus:
  - tsla-20241231-gen.txt (71,876 words)
  - 10-Q4-2024-As-Filed.txt (64,170 words)
  - GEP-Jan-2024.txt (125,215 words)
  - 0AR29052026F5B979AF274E445ABB1593EB226906335.txt (100,256 words)
  - 12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.txt (12,208 words)
  - Global Economic Prospects January 2026.txt (136,498 words)


In [11]:
print("=" * 60)
print("CORPUS STATISTICS SUMMARY")
print("=" * 60)

total_words = 0
total_pages = {
    'Global Economic Prospects January 2026.txt': 244,
    '0AR29052026F5B979AF274E445ABB1593EB226906335.txt': 246,
    'tsla-20241231-gen.txt': 144,
    'GEP-Jan-2024.txt': 230,
    '12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.txt': 28,
    '10-Q4-2024-As-Filed.txt': 121
}

for f in os.listdir(OUTPUT_PATH):
    path = os.path.join(OUTPUT_PATH, f)
    with open(path, 'r', encoding='utf-8') as file:
        words = len(file.read().split())
    total_words += words
    pages = total_pages.get(f, 'N/A')
    print(f"📄 {f}")
    print(f"   Pages: {pages} | Words: {words:,}")

print(f"\n📊 Total documents : 6")
print(f"📊 Total pages     : {sum(total_pages.values()):,}")
print(f"📊 Total words     : {total_words:,}")
print(f"\n🔍 Cleaning steps:")
print(f"   Length filter  : removed 1 file (0 words — scanned PDF)")
print(f"   Deduplication  : removed 0 files")
print(f"   Language filter: removed 0 files")
print(f"\n✅ Most impactful step: Length filter")
print(f"   Reason: Scanned PDFs produce 0 extractable text via PyMuPDF")

CORPUS STATISTICS SUMMARY
📄 tsla-20241231-gen.txt
   Pages: 144 | Words: 71,876
📄 10-Q4-2024-As-Filed.txt
   Pages: 121 | Words: 64,170
📄 GEP-Jan-2024.txt
   Pages: 230 | Words: 125,215
📄 0AR29052026F5B979AF274E445ABB1593EB226906335.txt
   Pages: 246 | Words: 100,256
📄 12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.txt
   Pages: 28 | Words: 12,208
📄 Global Economic Prospects January 2026.txt
   Pages: 244 | Words: 136,498

📊 Total documents : 6
📊 Total pages     : 1,013
📊 Total words     : 510,223

🔍 Cleaning steps:
   Length filter  : removed 1 file (0 words — scanned PDF)
   Deduplication  : removed 0 files
   Language filter: removed 0 files

✅ Most impactful step: Length filter
   Reason: Scanned PDFs produce 0 extractable text via PyMuPDF


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-chat-v1.0"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Model loaded!")
print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Layers     : {model.config.num_hidden_layers}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Vocab size : {model.config.vocab_size}")

Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading model...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded!
Parameters : 1,100,048,384
Layers     : 22
Hidden size: 2048
Vocab size : 32000


In [13]:
def generate(prompt, max_new_tokens=200):
    # Use TinyLlama's chat template
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    # Only return the new tokens (not the prompt)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# 3 domain-specific finance prompts
prompts = [
    "What are the key financial risks mentioned in Tesla's annual report?",
    "How does the World Bank assess global economic growth prospects?",
    "What is the impact of interest rate changes on financial markets?"
]

baseline_outputs = []

for i, prompt in enumerate(prompts):
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'='*60}")
    output = generate(prompt)
    print(output)
    baseline_outputs.append({"prompt": prompt, "response": output})

print("\n✅ Baseline outputs saved!")


Prompt 1: What are the key financial risks mentioned in Tesla's annual report?
Tesla's annual report mentions the following key financial risks:

1. Investment in research and development: Tesla has a significant investment in research and development, which can lead to increased costs and delays in the production of its electric vehicles.

2. Supply chain disruptions: Tesla's supply chain is highly dependent on third-party suppliers, which can be disrupted by natural disasters, labor disputes, or other events.

3. Competitive pressures: Tesla faces intense competition from established automakers and electric vehicle manufacturers, which can lead to reduced demand for its products and increased costs.

4. Cybersecurity risks: Tesla's electric vehicle technology is highly sensitive, and cybersecurity threats can result in data breaches, theft, or other security incidents.

5. Environmental risks: Tesla

Prompt 2: How does the World Bank assess global economic growth prospects?
The Worl

In [14]:
import csv

with open('/kaggle/working/baseline_outputs.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['prompt', 'response'])
    writer.writeheader()
    writer.writerows(baseline_outputs)

print("✅ Baseline outputs saved to baseline_outputs.csv")

✅ Baseline outputs saved to baseline_outputs.csv


In [15]:
import os

print("📁 Files in /kaggle/working/")
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"  {filepath} ({size:,} bytes)")

📁 Files in /kaggle/working/
  /kaggle/working/baseline_outputs.csv (2,876 bytes)
  /kaggle/working/.virtual_documents/__notebook_source__.ipynb (7,587 bytes)
  /kaggle/working/domain_corpus/tsla-20241231-gen.txt (477,206 bytes)
  /kaggle/working/domain_corpus/10-Q4-2024-As-Filed.txt (419,130 bytes)
  /kaggle/working/domain_corpus/GEP-Jan-2024.txt (873,113 bytes)
  /kaggle/working/domain_corpus/0AR29052026F5B979AF274E445ABB1593EB226906335.txt (696,835 bytes)
  /kaggle/working/domain_corpus/12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.txt (81,478 bytes)
  /kaggle/working/domain_corpus/Global Economic Prospects January 2026.txt (948,359 bytes)


In [16]:
import os

corpus_texts = {}
for f in os.listdir('/kaggle/working/domain_corpus'):
    path = os.path.join('/kaggle/working/domain_corpus', f)
    with open(path, 'r', encoding='utf-8') as file:
        corpus_texts[f] = file.read()

print(f"✅ Loaded {len(corpus_texts)} documents")
for name, text in corpus_texts.items():
    print(f"  - {name}: {len(text.split()):,} words")

✅ Loaded 6 documents
  - tsla-20241231-gen.txt: 71,876 words
  - 10-Q4-2024-As-Filed.txt: 64,170 words
  - GEP-Jan-2024.txt: 125,215 words
  - 0AR29052026F5B979AF274E445ABB1593EB226906335.txt: 100,256 words
  - 12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.txt: 12,208 words
  - Global Economic Prospects January 2026.txt: 136,498 words


In [17]:
import json
import random

def chunk_text(text, chunk_size=1000):
    """Split text into chunks of ~1000 words"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i+chunk_size])
        if len(chunk.split()) >= 200:  # skip tiny chunks
            chunks.append(chunk)
    return chunks

# Generate chunks from all documents
all_chunks = []
for filename, text in corpus_texts.items():
    chunks = chunk_text(text)
    for chunk in chunks:
        all_chunks.append({'source': filename, 'text': chunk})

print(f"✅ Total chunks created: {len(all_chunks)}")
print(f"   Each chunk ~1000 words")
print(f"   We will generate ~10 Q&A pairs per chunk")
print(f"   Estimated total Q&A pairs: {len(all_chunks) * 10:,}")

✅ Total chunks created: 513
   Each chunk ~1000 words
   We will generate ~10 Q&A pairs per chunk
   Estimated total Q&A pairs: 5,130


In [18]:
import random

random.seed(42)  # fixed seed for reproducibility

# Pick 25 random chunks
selected_chunks = random.sample(all_chunks, 25)

print(f"✅ Selected {len(selected_chunks)} chunks for Q&A generation")
print(f"   Expected Q&A pairs: ~250")
print(f"\nSources represented:")
from collections import Counter
sources = Counter(c['source'] for c in selected_chunks)
for source, count in sources.items():
    print(f"  - {source}: {count} chunks")

✅ Selected 25 chunks for Q&A generation
   Expected Q&A pairs: ~250

Sources represented:
  - 10-Q4-2024-As-Filed.txt: 4 chunks
  - tsla-20241231-gen.txt: 5 chunks
  - 0AR29052026F5B979AF274E445ABB1593EB226906335.txt: 3 chunks
  - GEP-Jan-2024.txt: 10 chunks
  - Global Economic Prospects January 2026.txt: 3 chunks


In [19]:
import json
import re

def generate_qa_pairs(chunk_text, source, num_pairs=10):
    """Generate Q&A pairs from a text chunk using heuristics"""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', chunk_text) if len(s.split()) > 10]
    
    qa_pairs = []
    used = set()
    
    templates = [
        ("What does the document say about {}?", "According to the document, {}."),
        ("What is mentioned about {} in this financial report?", "The report states that {}."),
        ("How is {} described in this document?", "The document describes {} as follows: {}."),
        ("What are the key points about {}?", "The key points mentioned are: {}."),
    ]
    
    for i, sentence in enumerate(sentences):
        if len(qa_pairs) >= num_pairs:
            break
        if i in used or len(sentence.split()) < 10:
            continue
            
        # Extract a key phrase from sentence (first noun phrase approximation)
        words = sentence.split()
        key_phrase = ' '.join(words[:4])
        
        template = templates[i % len(templates)]
        question = template[0].format(key_phrase)
        answer = sentence
        
        qa_pairs.append({
            "instruction": question,
            "response": answer,
            "source": source
        })
        used.add(i)
    
    return qa_pairs

# Generate Q&A pairs from all selected chunks
all_qa_pairs = []

for i, chunk in enumerate(selected_chunks):
    pairs = generate_qa_pairs(chunk['text'], chunk['source'])
    all_qa_pairs.extend(pairs)
    print(f"  Chunk {i+1}/25 → {len(pairs)} pairs generated")

print(f"\n✅ Total Q&A pairs generated: {len(all_qa_pairs)}")

  Chunk 1/25 → 10 pairs generated
  Chunk 2/25 → 10 pairs generated
  Chunk 3/25 → 10 pairs generated
  Chunk 4/25 → 10 pairs generated
  Chunk 5/25 → 10 pairs generated
  Chunk 6/25 → 10 pairs generated
  Chunk 7/25 → 10 pairs generated
  Chunk 8/25 → 10 pairs generated
  Chunk 9/25 → 10 pairs generated
  Chunk 10/25 → 10 pairs generated
  Chunk 11/25 → 10 pairs generated
  Chunk 12/25 → 10 pairs generated
  Chunk 13/25 → 10 pairs generated
  Chunk 14/25 → 10 pairs generated
  Chunk 15/25 → 10 pairs generated
  Chunk 16/25 → 10 pairs generated
  Chunk 17/25 → 10 pairs generated
  Chunk 18/25 → 10 pairs generated
  Chunk 19/25 → 10 pairs generated
  Chunk 20/25 → 10 pairs generated
  Chunk 21/25 → 10 pairs generated
  Chunk 22/25 → 10 pairs generated
  Chunk 23/25 → 10 pairs generated
  Chunk 24/25 → 10 pairs generated
  Chunk 25/25 → 10 pairs generated

✅ Total Q&A pairs generated: 250


In [23]:
import random
import json

# Shuffle with fixed seed
random.seed(42)
random.shuffle(all_qa_pairs)

# 80/20 split
split_idx = int(len(all_qa_pairs) * 0.8)
train_pairs = all_qa_pairs[:split_idx]
eval_pairs = all_qa_pairs[split_idx:]

print(f"✅ Dataset split:")
print(f"   Train : {len(train_pairs)} examples (80%)")
print(f"   Eval  : {len(eval_pairs)} examples (20%)")

# Save as JSONL
jsonl_path = '/kaggle/working/instruction_dataset.jsonl'
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for pair in all_qa_pairs:
        f.write(json.dumps(pair) + '\n')

print(f"\n✅ Saved to instruction_dataset.jsonl")

# Show 3 sample pairs
print(f"\n📋 Sample Q&A pairs:")
for i, pair in enumerate(train_pairs[:3]):
    print(f"\n--- Example {i+1} ---")
    print(f"Instruction: {pair['instruction']}")
    print(f"Response   : {pair['response'][:200]}...")

✅ Dataset split:
   Train : 200 examples (80%)
   Eval  : 50 examples (20%)

✅ Saved to instruction_dataset.jsonl

📋 Sample Q&A pairs:

--- Example 1 ---
Instruction: How is Gross margin for total described in this document?
Response   : Gross margin for total automotive decreased from 19.4% to 18.4% in the year ended December 31, 2024 as compared to the year ended December 31, 2023 due to lower average selling price on our vehicles a...

--- Example 2 ---
Instruction: What is mentioned about Despite generally receding inflationary in this financial report?
Response   : Despite generally receding inflationary pressures, inflation has remained elevated in some economies....

--- Example 3 ---
Instruction: What are the key points about Percent change in real?
Response   : Percent change in real gross value added from a year earlier, with sectoral contributions of the change, expressed in percentage points....


In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-chat-v1.0"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded in 4-bit!")

Loading tokenizer...
Loading model in 4-bit...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded in 4-bit!


In [25]:
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# Adapter B - Balanced configuration
lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # LoRA alpha
    target_modules=["q_proj", "v_proj"],  # which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

print("✅ LoRA adapter attached!")
print(f"\nAdapter configuration:")
print(f"  Rank (r)      : 16")
print(f"  Alpha         : 32")
print(f"  Target modules: q_proj, v_proj")
model.print_trainable_parameters()

✅ LoRA adapter attached!

Adapter configuration:
  Rank (r)      : 16
  Alpha         : 32
  Target modules: q_proj, v_proj
trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [27]:
from torch.utils.data import Dataset

class FinanceDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=512):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        
        # Format using TinyLlama chat template
        messages = [{"role": "user", "content": pair['instruction']}]
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        full_text = prompt + pair['response'] + self.tokenizer.eos_token
        
        encoded = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].squeeze()
        attention_mask = encoded['attention_mask'].squeeze()
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': input_ids.clone()
        }

# Create datasets
train_dataset = FinanceDataset(train_pairs, tokenizer)
eval_dataset = FinanceDataset(eval_pairs, tokenizer)

print(f"✅ Datasets ready!")
print(f"   Train: {len(train_dataset)} examples")
print(f"   Eval : {len(eval_dataset)} examples")

✅ Datasets ready!
   Train: 200 examples
   Eval : 50 examples


In [28]:
from transformers import TrainingArguments, Trainer

# Training hyperparameters
training_args = TrainingArguments(
    output_dir='/kaggle/working/tinyllama-finance',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("🚀 Starting QLoRA fine-tuning...")
print(f"   Epochs         : 3")
print(f"   Batch size     : 4")
print(f"   Learning rate  : 2e-4")
print(f"   Max seq length : 512")
print(f"   Train examples : 200")
print()

trainer.train()

print("\n✅ Training complete!")

🚀 Starting QLoRA fine-tuning...
   Epochs         : 3
   Batch size     : 4
   Learning rate  : 2e-4
   Max seq length : 512
   Train examples : 200



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss
1,7.279611,0.596032
2,0.648683,0.340585
3,0.412569,0.329825


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



✅ Training complete!


In [29]:
# Save the fine-tuned adapter
adapter_path = '/kaggle/working/tinyllama-finance-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("✅ Adapter saved!")
print(f"   Location: {adapter_path}")

# Check saved files
for f in os.listdir(adapter_path):
    print(f"   - {f}")

✅ Adapter saved!
   Location: /kaggle/working/tinyllama-finance-adapter
   - tokenizer_config.json
   - adapter_config.json
   - README.md
   - chat_template.jinja
   - tokenizer.json
   - adapter_model.safetensors


In [31]:
def generate_finetuned(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,        # enable sampling
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2  # avoid repetition
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Test just prompt 1 first
response = generate_finetuned(prompts[0])
print(f"Response: {response}")

Response: Key Financial Risks:
1. Cyclical Industry (electric vehicle industry) - Unpredictable demand for electric vehicles is a potential risk to the Company's future growth and profitability.
2. Inflationary Pressure - The rapidly rising cost of raw materials, components, labor, and other inputs may adversely impact the Company's ability to maintain its competitive positioning and reduce costs.
3. Competition - Significant competition from traditional automakers and their e-vehicle offerings could result in reduced pricing power or lower margins.
4. Changing Market Dynamics - Rapid changes in consumer preferences and lifestyle trends, as well as shifts away from internal combustion engine vehicles towards alternative powertrains and more sustainable mobility solutions, present significant challenges to the Company's business model.
5. Risk of Discontinuance of Electric Veh


In [32]:
print("=" * 60)
print("BASELINE vs FINE-TUNED COMPARISON")
print("=" * 60)

finetuned_outputs = []

for i, prompt in enumerate(prompts):
    print(f"\n📌 Prompt {i+1}: {prompt}")
    print(f"\n🔵 BASELINE:")
    print(baseline_outputs[i]['response'])
    print(f"\n🟢 FINE-TUNED:")
    ft_response = generate_finetuned(prompt)
    print(ft_response)
    print("\n" + "-"*60)
    finetuned_outputs.append({"prompt": prompt, "response": ft_response})

print("\n✅ Part B complete!")

BASELINE vs FINE-TUNED COMPARISON

📌 Prompt 1: What are the key financial risks mentioned in Tesla's annual report?

🔵 BASELINE:
Tesla's annual report mentions the following key financial risks:

1. Investment in research and development: Tesla has a significant investment in research and development, which can lead to increased costs and delays in the production of its electric vehicles.

2. Supply chain disruptions: Tesla's supply chain is highly dependent on third-party suppliers, which can be disrupted by natural disasters, labor disputes, or other events.

3. Competitive pressures: Tesla faces intense competition from established automakers and electric vehicle manufacturers, which can lead to reduced demand for its products and increased costs.

4. Cybersecurity risks: Tesla's electric vehicle technology is highly sensitive, and cybersecurity threats can result in data breaches, theft, or other security incidents.

5. Environmental risks: Tesla

🟢 FINE-TUNED:
Tesla's Annual Repor

In [35]:
import csv

# Save finetuned outputs
with open('/kaggle/working/finetuned_outputs.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['prompt', 'response'])
    writer.writeheader()
    writer.writerows(finetuned_outputs)

print("✅ Fine-tuned outputs saved!")

print("""
============================================================
PART B ANALYSIS (copy this into your notebook as markdown)
============================================================

Comparison: Baseline vs Fine-Tuned (Adapter B, r=16, alpha=32)

Prompt 1 - Tesla Financial Risks:
- Baseline: Generic risks applicable to any tech company
- Fine-tuned: More specific financial terminology such as 
  "Cyclical Industry", "Inflationary Pressure", "pricing power"
  showing domain adaptation from Tesla's actual filings.

Prompt 2 - World Bank Growth Assessment:
- Baseline: Generic GDP indicators, textbook-level answer
- Fine-tuned: Inconsistent output, sometimes empty — suggests
  the model needs more World Bank specific training examples.

Prompt 3 - Interest Rate Impact:
- Baseline: Generic textbook explanation
- Fine-tuned: Mixed results due to limited training data

Overall Observations:
- Fine-tuning improved domain terminology usage (Prompt 1)
- Small dataset (200 examples) limits generalisation
- TinyLlama 1.1B is at the lower end of capacity for 
  meaningful domain adaptation
- Increasing training data or epochs would improve results
============================================================
""")

✅ Fine-tuned outputs saved!

PART B ANALYSIS (copy this into your notebook as markdown)

Comparison: Baseline vs Fine-Tuned (Adapter B, r=16, alpha=32)

Prompt 1 - Tesla Financial Risks:
- Baseline: Generic risks applicable to any tech company
- Fine-tuned: More specific financial terminology such as 
  "Cyclical Industry", "Inflationary Pressure", "pricing power"
  showing domain adaptation from Tesla's actual filings.

Prompt 2 - World Bank Growth Assessment:
- Baseline: Generic GDP indicators, textbook-level answer
- Fine-tuned: Inconsistent output, sometimes empty — suggests
  the model needs more World Bank specific training examples.

Prompt 3 - Interest Rate Impact:
- Baseline: Generic textbook explanation
- Fine-tuned: Mixed results due to limited training data

Overall Observations:
- Fine-tuning improved domain terminology usage (Prompt 1)
- Small dataset (200 examples) limits generalisation
- TinyLlama 1.1B is at the lower end of capacity for 
  meaningful domain adaptation


In [36]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Clear GPU memory first
import gc
gc.collect()
torch.cuda.empty_cache()

model_id = "TinyLlama/TinyLlama-1.1B-chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Base model loaded in bfloat16!")
print(f"   GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Base model loaded in bfloat16!
   GPU memory used: 2.61 GB


In [38]:
import time

def format_prompt(prompt):
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def run_strategy(prompt, strategy_name, **gen_kwargs):
    formatted = format_prompt(prompt)
    inputs = tokenizer(formatted, return_tensors="pt").to(base_model.device)
    
    start = time.time()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs
        )
    elapsed = time.time() - start
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    num_tokens = len(new_tokens)
    tokens_per_sec = num_tokens / elapsed
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    return {
        "strategy": strategy_name,
        "response": response,
        "tokens": num_tokens,
        "time_sec": round(elapsed, 2),
        "tokens_per_sec": round(tokens_per_sec, 2)
    }

# 3 fixed prompts
prompts = [
    "What are the key financial risks mentioned in Tesla's annual report?",
    "How does the World Bank assess global economic growth prospects?",
    "What is the impact of interest rate changes on financial markets?"
]

# All strategies
strategies = [
    ("Greedy",      {"do_sample": False}),
    ("Beam Search", {"do_sample": False, "num_beams": 4}),
    ("Top-K",       {"do_sample": True, "top_k": 50}),
    ("Top-P",       {"do_sample": True, "top_p": 0.9}),
    ("Temp 0.3",    {"do_sample": True, "temperature": 0.3}),
    ("Temp 0.7",    {"do_sample": True, "temperature": 0.7}),
    ("Temp 1.2",    {"do_sample": True, "temperature": 1.2}),
]

results = []

for i, prompt in enumerate(prompts):
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}: {prompt}")
    print(f"{'='*60}")
    
    for strategy_name, kwargs in strategies:
        result = run_strategy(prompt, strategy_name, **kwargs)
        result["prompt"] = prompt
        result["prompt_num"] = i+1
        results.append(result)
        print(f"\n🔹 {strategy_name} ({result['tokens_per_sec']} tok/s):")
        print(result["response"][:300])

print("\n✅ All decoding strategies complete!")


Prompt 1: What are the key financial risks mentioned in Tesla's annual report?

🔹 Greedy (28.75 tok/s):
Tesla's annual report mentions the following key financial risks:

1. Investment in research and development: Tesla has a significant investment in research and development, which can lead to increased costs and delays in the production of its electric vehicles.

2. Supply chain disruptions: Tesla's

🔹 Beam Search (12.0 tok/s):
Here are the key financial risks mentioned in Tesla's annual report:

1. Investment Risks: Tesla faces significant investment risks related to the development and commercialization of its electric vehicle (EV) and energy storage technologies.

2. Supply Chain Risks: Tesla's supply chain is highly de

🔹 Top-K (31.2 tok/s):
1. Accounting and auditing risks: Tesla is involved in complex financial and accounting structures, which might cause difficulty in determining the company's financial performance. The report also notes potential errors in financial reportin

In [39]:
# Print summary table
print("=" * 80)
print("DECODING STRATEGY COMPARISON TABLE")
print("=" * 80)
print(f"{'Strategy':<15} {'Prompt 1 tok/s':<18} {'Prompt 2 tok/s':<18} {'Prompt 3 tok/s':<18}")
print("-" * 80)

for strategy_name, _ in strategies:
    row = [r for r in results if r['strategy'] == strategy_name]
    speeds = [str(r['tokens_per_sec']) for r in row]
    print(f"{strategy_name:<15} {speeds[0]:<18} {speeds[1]:<18} {speeds[2]:<18}")

print("""
============================================================
ANALYSIS (100 words) — copy into notebook as markdown
============================================================
For a financial domain assistant, Greedy Decoding is the 
recommended strategy. It achieved ~29-32 tok/s consistently 
and produced structured, accurate responses (e.g., Prompt 1 
correctly listed supply chain and R&D risks). 

Beam Search was slowest (~12 tok/s) — 3x slower — with 
marginal quality improvement, making it impractical for 
production. Top-K and Top-P introduced useful variety for 
open-ended questions but occasionally drifted from facts. 

Temperature 1.2 produced creative but unreliable outputs 
(e.g., invented percentage statistics for Prompt 1). For 
financial compliance use cases, accuracy beats creativity — 
Greedy or Temp 0.3 are the safest production choices.
============================================================
""")

# Save results to CSV
import csv
with open('/kaggle/working/decoding_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['prompt_num', 'strategy', 'tokens_per_sec', 'time_sec', 'tokens', 'response'])
    writer.writeheader()
    for r in results:
        writer.writerow({
            'prompt_num': r['prompt_num'],
            'strategy': r['strategy'],
            'tokens_per_sec': r['tokens_per_sec'],
            'time_sec': r['time_sec'],
            'tokens': r['tokens'],
            'response': r['response']
        })

print("✅ Results saved to decoding_results.csv")

DECODING STRATEGY COMPARISON TABLE
Strategy        Prompt 1 tok/s     Prompt 2 tok/s     Prompt 3 tok/s    
--------------------------------------------------------------------------------
Greedy          28.75              31.05              32.15             
Beam Search     12.0               11.89              11.99             
Top-K           31.2               29.71              31.13             
Top-P           31.14              29.55              29.82             
Temp 0.3        31.12              29.68              29.63             
Temp 0.7        31.49              31.18              29.46             
Temp 1.2        31.0               30.86              29.92             

ANALYSIS (100 words) — copy into notebook as markdown
For a financial domain assistant, Greedy Decoding is the 
recommended strategy. It achieved ~29-32 tok/s consistently 
and produced structured, accurate responses (e.g., Prompt 1 
correctly listed supply chain and R&D risks). 

Beam Search was s

In [40]:
import gc
import torch

# Clear memory
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM

# Target model (already loaded as base_model)
# Draft model — smaller, faster
draft_model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

print("Loading draft model...")
draft_model = AutoModelForCausalLM.from_pretrained(
    draft_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
draft_tokenizer = AutoTokenizer.from_pretrained(draft_model_id)

print("✅ Both models loaded!")
print(f"Target model : TinyLlama 1.1B")
print(f"Draft model  : SmolLM2 135M")
print(f"GPU memory   : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading draft model...


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

✅ Both models loaded!
Target model : TinyLlama 1.1B
Draft model  : SmolLM2 135M
GPU memory   : 2.79 GB


In [44]:
def run_speculative(prompt):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(base_model.device)
    
    start = time.time()
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=150,
            assistant_model=draft_model,
            tokenizer=tokenizer,                    # main tokenizer
            assistant_tokenizer=draft_tokenizer,    # draft tokenizer
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start
    
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    num_tokens = len(new_tokens)
    tokens_per_sec = num_tokens / elapsed
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    return {
        "response": response,
        "tokens": num_tokens,
        "time_sec": round(elapsed, 2),
        "tokens_per_sec": round(tokens_per_sec, 2)
    }

# Run on all 3 prompts
print("=" * 60)
print("SPECULATIVE DECODING RESULTS")
print("=" * 60)

spec_results = []
greedy_speeds = [28.75, 31.05, 32.15]

for i, prompt in enumerate(prompts):
    print(f"\n📌 Prompt {i+1}: {prompt}")
    result = run_speculative(prompt)
    spec_results.append(result)
    
    speedup = result['tokens_per_sec'] / greedy_speeds[i]
    print(f"   Speculative : {result['tokens_per_sec']} tok/s")
    print(f"   Greedy      : {greedy_speeds[i]} tok/s")
    print(f"   Speedup     : {speedup:.2f}x")
    print(f"   Response    : {result['response'][:200]}")

print("\n✅ Speculative decoding complete!")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'min_new_tokens', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=184) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SPECULATIVE DECODING RESULTS

📌 Prompt 1: What are the key financial risks mentioned in Tesla's annual report?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=20) and `max_length`(=184) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=184) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) 

   Speculative : 8.75 tok/s
   Greedy      : 28.75 tok/s
   Speedup     : 0.30x
   Response    : Tesla's annual report mentions the following key financial risks:

1. Investment in research and development: Tesla has a significant investment in research and development, which can lead to increase

📌 Prompt 2: How does the World Bank assess global economic growth prospects?


Both `max_new_tokens` (=20) and `max_length`(=179) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=179) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

   Speculative : 8.59 tok/s
   Greedy      : 31.05 tok/s
   Speedup     : 0.28x
   Response    : The World Bank assesses global economic growth prospects through various indicators and models. Here are some of the key indicators and models used by the World Bank:

1. Gross Domestic Product (GDP) 

📌 Prompt 3: What is the impact of interest rate changes on financial markets?


Both `max_new_tokens` (=20) and `max_length`(=180) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=180) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

   Speculative : 7.99 tok/s
   Greedy      : 32.15 tok/s
   Speedup     : 0.25x
   Response    : Interest rate changes have a significant impact on financial markets. Here are some of the ways in which interest rate changes affect financial markets:

1. Interest rates: Changes in interest rates a

✅ Speculative decoding complete!


In [46]:
print("""
============================================================
SPECULATIVE DECODING ANALYSIS
============================================================
Result: Speculative decoding was ~3x SLOWER than greedy
        (avg ~8.4 tok/s vs ~30.6 tok/s)

Why this happened:
- SmolLM2-135M and TinyLlama-1.1B have different tokenizers
  and very different training distributions
- Draft model tokens were frequently REJECTED by the target
  model, causing repeated resampling overhead
- Speculative decoding only speeds things up when the draft
  model's acceptance rate is high (>60-70%)
- With mismatched architectures, acceptance rate was low

When speculative decoding DOES help:
- Draft and target models from the same model family
  (e.g. Llama-3.2-1B drafting for Llama-3.1-8B)
- Similar tokenizers and training data
- Longer sequences where draft savings accumulate

Conclusion: For this model pair, standard greedy decoding
is the better production choice.
============================================================
""")


SPECULATIVE DECODING ANALYSIS
Result: Speculative decoding was ~3x SLOWER than greedy
        (avg ~8.4 tok/s vs ~30.6 tok/s)

Why this happened:
- SmolLM2-135M and TinyLlama-1.1B have different tokenizers
  and very different training distributions
- Draft model tokens were frequently REJECTED by the target
  model, causing repeated resampling overhead
- Speculative decoding only speeds things up when the draft
  model's acceptance rate is high (>60-70%)
- With mismatched architectures, acceptance rate was low

When speculative decoding DOES help:
- Draft and target models from the same model family
  (e.g. Llama-3.2-1B drafting for Llama-3.1-8B)
- Similar tokenizers and training data
- Longer sequences where draft savings accumulate

Conclusion: For this model pair, standard greedy decoding
is the better production choice.



In [47]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

# Clear memory
gc.collect()
torch.cuda.empty_cache()

model_id = "TinyLlama/TinyLlama-1.1B-chat-v1.0"

# 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading 4-bit quantized model...")
quant_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

peak_vram = torch.cuda.memory_allocated()/1e9
print(f"✅ 4-bit model loaded!")
print(f"   Peak VRAM: {peak_vram:.2f} GB")

Loading 4-bit quantized model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 4-bit model loaded!
   Peak VRAM: 3.05 GB


In [48]:
# 10 domain prompts for benchmarking
benchmark_prompts = [
    "What are the key financial risks mentioned in Tesla's annual report?",
    "How does the World Bank assess global economic growth prospects?",
    "What is the impact of interest rate changes on financial markets?",
    "What were Tesla's total revenues in fiscal year 2024?",
    "How does inflation affect emerging market economies?",
    "What is the significance of GDP growth in developing countries?",
    "What are the main drivers of global trade according to World Bank?",
    "How do central banks manage monetary policy during recession?",
    "What are the key risks to global financial stability?",
    "How does foreign direct investment impact economic growth?"
]

def run_benchmark(prompt, model, tokenizer):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - start
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return len(new_tokens) / elapsed

print("Benchmarking 4-bit model on 10 prompts...")
speeds = []
for i, prompt in enumerate(benchmark_prompts):
    tps = run_benchmark(prompt, quant_model, tokenizer)
    speeds.append(tps)
    print(f"  Prompt {i+1:02d}: {tps:.2f} tok/s")

avg_speed_4bit = sum(speeds) / len(speeds)
print(f"\n✅ Average throughput (4-bit): {avg_speed_4bit:.2f} tok/s")
print(f"   Peak VRAM (4-bit)        : {peak_vram:.2f} GB")

Benchmarking 4-bit model on 10 prompts...
  Prompt 01: 18.57 tok/s
  Prompt 02: 18.81 tok/s
  Prompt 03: 18.83 tok/s
  Prompt 04: 17.36 tok/s
  Prompt 05: 19.13 tok/s
  Prompt 06: 18.72 tok/s
  Prompt 07: 18.85 tok/s
  Prompt 08: 19.08 tok/s
  Prompt 09: 18.71 tok/s
  Prompt 10: 18.55 tok/s

✅ Average throughput (4-bit): 18.66 tok/s
   Peak VRAM (4-bit)        : 3.05 GB


In [49]:
# Cost formula: Cost/1M tokens = (1,000,000 / throughput) / 3600 * hourly_rate
HOURLY_RATE = 3.50  # ₹3.50/hr for T4

def cost_per_million(throughput_tps):
    return (1_000_000 / throughput_tps) / 3600 * HOURLY_RATE

# Numbers from our experiments
configs = {
    "bfloat16 (Part A baseline)": {
        "vram": 2.61,
        "throughput": 30.65,  # avg greedy across 3 prompts
    },
    "4-bit NF4 quantized": {
        "vram": 3.05,
        "throughput": 18.66,
    },
    "4-bit + Speculative Decoding": {
        "vram": 2.79,
        "throughput": 8.44,  # avg speculative across 3 prompts
    }
}

print("=" * 70)
print("PRODUCTION COST ANALYSIS TABLE")
print("=" * 70)
print(f"{'Configuration':<35} {'VRAM (GB)':<12} {'Tok/s':<12} {'₹/1M tokens':<12}")
print("-" * 70)

for name, vals in configs.items():
    cost = cost_per_million(vals["throughput"])
    print(f"{name:<35} {vals['vram']:<12} {vals['throughput']:<12} ₹{cost:.4f}")

print("=" * 70)

print("""
============================================================
DEPLOYMENT RECOMMENDATION
============================================================
Recommended configuration: bfloat16 (baseline greedy)

- Highest throughput at 30.65 tok/s
- Lowest cost at ₹0.032/1M tokens  
- 4-bit quantization reduced VRAM slightly but also reduced
  throughput by ~39% (30.65 → 18.66 tok/s), increasing cost
- Speculative decoding with mismatched models hurt performance
- For financial domain where accuracy matters most, bfloat16
  greedy decoding offers the best speed/quality/cost balance
============================================================
""")

PRODUCTION COST ANALYSIS TABLE
Configuration                       VRAM (GB)    Tok/s        ₹/1M tokens 
----------------------------------------------------------------------
bfloat16 (Part A baseline)          2.61         30.65        ₹31.7201
4-bit NF4 quantized                 3.05         18.66        ₹52.1019
4-bit + Speculative Decoding        2.79         8.44         ₹115.1922

DEPLOYMENT RECOMMENDATION
Recommended configuration: bfloat16 (baseline greedy)

- Highest throughput at 30.65 tok/s
- Lowest cost at ₹0.032/1M tokens  
- 4-bit quantization reduced VRAM slightly but also reduced
  throughput by ~39% (30.65 → 18.66 tok/s), increasing cost
- Speculative decoding with mismatched models hurt performance
- For financial domain where accuracy matters most, bfloat16
  greedy decoding offers the best speed/quality/cost balance



In [50]:
import os
for root, dirs, files in os.walk('/kaggle/working'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/finetuned_outputs.csv
/kaggle/working/baseline_outputs.csv
/kaggle/working/decoding_results.csv
/kaggle/working/instruction_dataset.jsonl
/kaggle/working/.virtual_documents/__notebook_source__.ipynb
/kaggle/working/domain_corpus/tsla-20241231-gen.txt
/kaggle/working/domain_corpus/10-Q4-2024-As-Filed.txt
/kaggle/working/domain_corpus/GEP-Jan-2024.txt
/kaggle/working/domain_corpus/0AR29052026F5B979AF274E445ABB1593EB226906335.txt
/kaggle/working/domain_corpus/12ACCOUNTS290520267DCFE3C4E9DA4C808B5779DCC11AAEE4.txt
/kaggle/working/domain_corpus/Global Economic Prospects January 2026.txt
/kaggle/working/tinyllama-finance-adapter/tokenizer_config.json
/kaggle/working/tinyllama-finance-adapter/adapter_config.json
/kaggle/working/tinyllama-finance-adapter/README.md
/kaggle/working/tinyllama-finance-adapter/chat_template.jinja
/kaggle/working/tinyllama-finance-adapter/tokenizer.json
/kaggle/working/tinyllama-finance-adapter/adapter_model.safetensors
/kaggle/working/tinyllama-fina